In [ ]:
# %% [markdown]
# # Verify: OrganicFlowPhase split into Departure + Arrival
#
# Blocks:
# 1. Imports and path bootstrap
# 2. Load data: Mock -> Raw -> Resolved
# 3. Run combined (OrganicFlowPhase) simulation
# 4. Run split (OrganicDeparturePhase + OrganicArrivalPhase) simulation
# 5. Flow parity check
# 6. Inventory parity check
# 7. Cross-check: split vs combined inventory logs

In [ ]:
# %% 1. Imports and path bootstrap
import sys
from pathlib import Path

for _candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (_candidate / "gbp").is_dir() and (_candidate / "tests").is_dir():
        sys.path.insert(0, str(_candidate))
        break

import pandas as pd

from gbp.build.pipeline import build_model
from gbp.consumers.simulator import (
    Environment,
    EnvironmentConfig,
    OrganicArrivalPhase,
    OrganicDeparturePhase,
    OrganicFlowPhase,
)
from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

In [ ]:
# %% 2. Load data: Mock -> Raw -> Resolved
mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 168,
    "time_freq": "h",
    "start_date": "2025-01-01",
    "num_resources": 3,
    "resource_capacity": 100,
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}

mock = DataLoaderMock(mock_config)
loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()
resolved = build_model(raw)

In [ ]:
# %% 3. Run combined (OrganicFlowPhase) simulation
combined_log = Environment(
    resolved,
    EnvironmentConfig(phases=[OrganicFlowPhase()], seed=42, scenario_id="combined"),
).run()
combined_tables = combined_log.to_dataframes()

In [ ]:
# %% 4. Run split (OrganicDeparturePhase + OrganicArrivalPhase) simulation
split_log = Environment(
    resolved,
    EnvironmentConfig(
        phases=[OrganicDeparturePhase(), OrganicArrivalPhase()],
        seed=42,
        scenario_id="split",
    ),
).run()
split_tables = split_log.to_dataframes()

In [ ]:
# %% 5. Flow parity check — split flow_log vs resolved.observed_flow
flow_keys = ["period_id", "source_id", "target_id", "commodity_category"]

simulated_flow = (
    split_tables["simulation_flow_log"]
    .groupby(flow_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "simulated_quantity"})
)
observed_flow = (
    resolved.observed_flow
    .groupby(flow_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "observed_quantity"})
)
flow_compare = simulated_flow.merge(observed_flow, on=flow_keys, how="outer")
flow_compare["simulated_quantity"] = flow_compare["simulated_quantity"].fillna(0.0)
flow_compare["observed_quantity"] = flow_compare["observed_quantity"].fillna(0.0)
flow_compare["diff"] = flow_compare["simulated_quantity"] - flow_compare["observed_quantity"]
flow_mismatches = flow_compare.loc[flow_compare["diff"].abs() > 1e-9].reset_index(drop=True)

In [ ]:
# %% 6. Inventory parity check — split inventory_log vs resolved.observed_inventory
inventory_keys = ["period_id", "facility_id", "commodity_category"]
observed_facility_ids = resolved.observed_inventory["facility_id"].unique()

simulated_inventory = (
    split_tables["simulation_inventory_log"]
    .loc[lambda d: d["facility_id"].isin(observed_facility_ids)]
    .groupby(inventory_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "simulated_quantity"})
)
observed_inventory = (
    resolved.observed_inventory
    .groupby(inventory_keys, as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "observed_quantity"})
)
inventory_compare = simulated_inventory.merge(observed_inventory, on=inventory_keys, how="outer")
inventory_compare["simulated_quantity"] = inventory_compare["simulated_quantity"].fillna(0.0)
inventory_compare["observed_quantity"] = inventory_compare["observed_quantity"].fillna(0.0)
inventory_compare["diff"] = inventory_compare["simulated_quantity"] - inventory_compare["observed_quantity"]
inventory_mismatches = inventory_compare.loc[inventory_compare["diff"].abs() > 1e-9].reset_index(drop=True)

In [ ]:
# %% 7. Cross-check: split vs combined inventory logs must be identical
combined_inv = (
    combined_tables["simulation_inventory_log"]
    .sort_values(["period_id", "facility_id", "commodity_category"])
    .reset_index(drop=True)
)
split_inv = (
    split_tables["simulation_inventory_log"]
    .sort_values(["period_id", "facility_id", "commodity_category"])
    .reset_index(drop=True)
)
cross_check_ok = combined_inv[["period_id", "facility_id", "commodity_category", "quantity"]].equals(
    split_inv[["period_id", "facility_id", "commodity_category", "quantity"]]
)

results = {
    "flow_mismatches_count": len(flow_mismatches),
    "inventory_mismatches_count": len(inventory_mismatches),
    "split_vs_combined_inventory_identical": cross_check_ok,
}

In [ ]:
results